# B2 · mantenimiento y llantas MX/US
## tl;dr
Dos contrastes modelados reproducibles por vehículo-km; no costo observado del piloto, no importes adoptados. La tabla separa monedas/años y no permite compararlos directamente ni sumarlos para D2D.


## Context & Methods
### Key Assumptions
IMT: modelo técnico T3S2 a precios 2025, tabla 4.2, PDF física 61. TTI: estimaciones 2024 para valor de demora, Exhibit C-3, PDF física 7 y método física 8. ATRI 2026: formulario metodológico, no precios ni prueba de idéntica frontera TTI. Un galón no interviene: USD/vehículo-milla ÷ 1.609344 = USD/vehículo-km. MXN por 1000 vehículo-km ÷ 1000. FX no elegido.
Se conserva el costo publicado, no se reconstruye con cantidades/coefs redondeados. El recape es secuencial, no otra rueda. Tracto/remolque no desagregados; sin reparto inventado.
[IMT](https://imt.mx/archivos/Publicaciones/PublicacionTecnica/pt881.pdf) · [TTI](https://static.tti.tamu.edu/tti.tamu.edu/documents/mobility-report-2025-appx-c.pdf) · [ATRI metodología](https://truckingresearch.org/wp-content/uploads/2026/02/2026-Ops-Cost-Motor-Carrier-Form.pdf)


## Data
### 1. Leer manifiesto y verificar archivos
Desde la raíz o una subcarpeta del worktree. Sólo archivos locales; sin SQL, credenciales ni acceso a DB. Diez hashes: tres PDF y siete expedientes previos. Los originales permiten inspeccionar fuente/contexto, no hacen aplicable el modelo a otra flota.


In [ ]:
import json, hashlib, math
from pathlib import Path
manifest = "docs/research/B2_MAINTENANCE_TIRES_REFERENCE_2026-09-30.json"
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / manifest).is_file())
data = json.loads((root / manifest).read_text(encoding="utf-8"))
for pin in data["unchangedFiles"] + data["sources"]:
    assert hashlib.sha256((root / pin["file"]).read_bytes()).hexdigest() == pin["sha256"]
assert data["adoptedValues"] == 0 and not data["calibrated"]
assert all(c["maintenanceUsdPerKm"] is None and c["tiresUsdPerKm"] is None for c in data["countryCards"])



## Results
### 2. Normalizar y conciliar
La suma mexicana agrega labor/refacciones/lubricantes/llantas del mismo modelo. La suma US conserva únicamente las dos líneas publicadas; lubricantes no identificados por separado no significan costo cero. Ninguna es CVU completo. Se usa una tabla exacta de dos fichas; una gráfica monetaria MX↔US sería una comparación inválida.


In [ ]:
rows = {r["id"]: r for r in data["records"]}
mx_maintenance = (rows["MX_LABOR"]["value"] + rows["MX_PARTS"]["value"]) / 1000
mx_lubes = rows["MX_LUBRICANTS"]["value"] / 1000
mx_tires = rows["MX_TIRES"]["value"] / 1000
us_maintenance = rows["US_MAINT"]["value"] / 1.609344
us_tires = rows["US_TIRES"]["value"] / 1.609344
assert math.isclose(mx_maintenance, 5.61754, abs_tol=1e-12)
assert math.isclose(mx_maintenance + mx_lubes + mx_tires, 6.86717, abs_tol=1e-12)
assert math.isclose(us_maintenance + us_tires, 0.26 / 1.609344, abs_tol=1e-12)
assert data["groups"][0]["sourceId"] != data["groups"][1]["sourceId"]
assert all(not g["countryComparisonAllowed"] and not g["targetMatched"] for g in data["groups"])
mx_printed = data["printedPrecision"]["mx"]
assert not math.isclose(mx_printed["tireEquivalentNewPer1000KmRounded"] * mx_printed["newTireMxn"], rows["MX_TIRES"]["value"], abs_tol=0.01)
assert not math.isclose(mx_printed["partsPercentOfVehicleRounded"] / 100 * mx_printed["vehiclePriceMxn"], rows["MX_PARTS"]["value"], abs_tol=0.01)
assert not data["sharedTrailer"]["arbitraryHalfSplitAllowed"]



### 3. Tabla exacta con unidades y fechas


In [ ]:
print(json.dumps({"status": "PASS", "scope": "SOURCE_SELECTED_CONTRAST_NOT_COMPLETE_CVU", "matchedFiles": 10,
    "cards": [
        {"country": "MX", "costYear": "2025", "currency": "MXN", "maintenancePublishedComponentsWithoutTires": mx_maintenance,
         "separateLubricants": mx_lubes, "tires": mx_tires, "selectedSubtotalPerVehicleKm": mx_maintenance + mx_lubes + mx_tires},
        {"country": "US", "costYear": "2024", "currency": "USD", "maintenancePublishedComponentsWithoutTires": us_maintenance,
         "separateLubricants": None, "tires": us_tires, "selectedSubtotalPerVehicleKm": us_maintenance + us_tires}],
    "pilotRates": None, "adopted": 0, "jupyterKernelExecuted": False}, indent=2))



{
  "status": "PASS",
  "scope": "SOURCE_SELECTED_CONTRAST_NOT_COMPLETE_CVU",
  "matchedFiles": 10,
  "cards": [
    {
      "country": "MX",
      "costYear": "2025",
      "currency": "MXN",
      "maintenancePublishedComponentsWithoutTires": 5.61754,
      "separateLubricants": 0.27967000000000003,
      "tires": 0.96996,
      "selectedSubtotalPerVehicleKm": 6.867170000000001
    },
    {
      "country": "US",
      "costYear": "2024",
      "currency": "USD",
      "maintenancePublishedComponentsWithoutTires": 0.1317306927543148,
      "separateLubricants": null,
      "tires": 0.02982581722739203,
      "selectedSubtotalPerVehicleKm": 0.16155650998170681
    }
  ],
  "pilotRates": null,
  "adopted": 0,
  "jupyterKernelExecuted": false
}



## Takeaways
No se descompone un agregado en los 31 inputs PM/reservas ni los 12 de llantas. No se usan defaults como benchmark. Estimaciones públicas no son compra ni venta XBF.
Siguiente: consolidar fijo/exposición/remolque para reunir una ficha de referencia aplicable o un faltante acotado, y pasar al caso D2D. Mantenimiento/llantas sin calibrar; 0/4 gates.
### Validación pendiente de presentación
Cuerpo de celdas validable con Python estándar; kernel Jupyter y visualización renderizada pendientes. Para validar con las dependencias Jupyter disponibles: ejecutar desde la raíz `python -m jupyter nbconvert --execute --to notebook --inplace docs/research/B2_MAINTENANCE_TIRES_REFERENCE_2026-09-30.ipynb`, luego abrir el notebook y revisar summary/results/table. No requiere acceso externo.
